First-Touch Attribution

In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("cleaned_multi__touch_attribution_dataset.csv")

df["event_timestamp_utc"] = pd.to_datetime(df["event_timestamp_utc"])

df.head()

,event_id,user_id,journey_id,session_id,event_timestamp_utc,event_date,touchpoint_number,total_touchpoints_in_journey,channel,channel_group,...,landing_page,device,region,product_plan,ad_spend,is_conversion,conversion_id,conversion_timestamp_utc,conversion_value,customer_status
0,E000001,U00001,J00001,S00001_01,2026-02-05 11:14:00,2026-02-05,1,4,TikTok Ads,paid_social,...,/free-trial,Mobile,South,Professional,0.20,0,NaN,NaN,0.00,NaN
1,E000002,U00001,J00001,S00001_02,2026-02-07 02:26:00,2026-02-07,2,4,Organic Search,organic,...,/demo,Mobile,South,Professional,0.00,0,NaN,NaN,0.00,NaN
2,E000003,U00001,J00001,S00001_03,2026-02-08 00:17:00,2026-02-08,3,4,Direct,direct,...,/blog/marketing-roi,Mobile,South,Professional,0.00,0,NaN,NaN,0.00,NaN
3,E000004,U00001,J00001,S00001_04,2026-02-08 18:05:00,2026-02-08,4,4,Meta Ads,paid_social,...,/blog/marketing-roi,Mobile,South,Professional,0.62,1,C00001,2026-02-08 18:05:00,267.15,returning
4,E000005,U00002,J00002,S00002_01,2026-03-10 09:59:00,2026-03-10,1,5,LinkedIn Ads,paid_social,...,/pricing,Desktop,West,Starter,3.75,0,NaN,NaN,0.00,NaN


In [4]:
conversion_rows = df[df["is_conversion"] == 1]

conversion_rows.head()

,event_id,user_id,journey_id,session_id,event_timestamp_utc,event_date,touchpoint_number,total_touchpoints_in_journey,channel,channel_group,...,landing_page,device,region,product_plan,ad_spend,is_conversion,conversion_id,conversion_timestamp_utc,conversion_value,customer_status
3,E000004,U00001,J00001,S00001_04,2026-02-08 18:05:00,2026-02-08,4,4,Meta Ads,paid_social,...,/blog/marketing-roi,Mobile,South,Professional,0.62,1,C00001,2026-02-08 18:05:00,267.15,returning
13,E000014,U00004,J00004,S00004_04,2026-03-21 05:31:00,2026-03-21,4,4,Email,owned,...,/demo,Desktop,Central,Starter,0.08,1,C00002,2026-03-21 05:31:00,109.37,returning
19,E000020,U00006,J00006,S00006_04,2026-04-07 14:20:00,2026-04-07,4,4,Email,owned,...,/case-studies,Tablet,North,Professional,0.02,1,C00003,2026-04-07 14:20:00,352.78,new
27,E000028,U00008,J00008,S00008_05,2026-01-28 08:23:00,2026-01-28,5,5,Organic Search,organic,...,/pricing,Desktop,West,Starter,0.00,1,C00004,2026-01-28 08:23:00,127.44,new
39,E000040,U00011,J00011,S00011_02,2026-02-10 19:58:00,2026-02-10,2,2,Google Ads,paid_search,...,/case-studies,Mobile,North,Starter,3.18,1,C00005,2026-02-10 19:58:00,89.24,new


In [5]:
converted_journey_ids = conversion_rows["journey_id"].unique()

converted_journey_ids[:10]

array(['J00001', 'J00004', 'J00006', 'J00008', 'J00011', 'J00022',
       'J00023', 'J00025', 'J00026', 'J00027'], dtype=object)

In [6]:
len(converted_journey_ids)

209

In [7]:
converted_df = df[df["journey_id"].isin(converted_journey_ids)].copy()

converted_df.head()

,event_id,user_id,journey_id,session_id,event_timestamp_utc,event_date,touchpoint_number,total_touchpoints_in_journey,channel,channel_group,...,landing_page,device,region,product_plan,ad_spend,is_conversion,conversion_id,conversion_timestamp_utc,conversion_value,customer_status
0,E000001,U00001,J00001,S00001_01,2026-02-05 11:14:00,2026-02-05,1,4,TikTok Ads,paid_social,...,/free-trial,Mobile,South,Professional,0.20,0,NaN,NaN,0.00,NaN
1,E000002,U00001,J00001,S00001_02,2026-02-07 02:26:00,2026-02-07,2,4,Organic Search,organic,...,/demo,Mobile,South,Professional,0.00,0,NaN,NaN,0.00,NaN
2,E000003,U00001,J00001,S00001_03,2026-02-08 00:17:00,2026-02-08,3,4,Direct,direct,...,/blog/marketing-roi,Mobile,South,Professional,0.00,0,NaN,NaN,0.00,NaN
3,E000004,U00001,J00001,S00001_04,2026-02-08 18:05:00,2026-02-08,4,4,Meta Ads,paid_social,...,/blog/marketing-roi,Mobile,South,Professional,0.62,1,C00001,2026-02-08 18:05:00,267.15,returning
10,E000011,U00004,J00004,S00004_01,2026-03-16 14:57:00,2026-03-16,1,4,Google Ads,paid_search,...,/demo,Desktop,Central,Starter,1.13,0,NaN,NaN,0.00,NaN


In [8]:
converted_df.shape

(838, 26)

In [9]:
first_touch = converted_df.sort_values("event_timestamp_utc").groupby("journey_id").first().reset_index()

first_touch[["journey_id", "channel", "campaign", "event_timestamp_utc"]].head()

,journey_id,channel,campaign,event_timestamp_utc
0,J00001,TikTok Ads,creator_spark,2026-02-05 11:14:00
1,J00004,Google Ads,brand_search,2026-03-16 14:57:00
2,J00006,Referral,affiliate_blog,2026-04-04 15:52:00
3,J00008,Google Ads,high_intent_keywords,2026-01-25 11:34:00
4,J00011,Organic Search,comparison_page,2026-02-09 13:57:00


In [10]:
journey_revenue = converted_df.groupby("journey_id")["conversion_value"].max().reset_index()

journey_revenue = journey_revenue.rename(
    columns={"conversion_value": "journey_revenue"}
)

journey_revenue.head()

,journey_id,journey_revenue
0,J00001,267.15
1,J00004,109.37
2,J00006,352.78
3,J00008,127.44
4,J00011,89.24


In [11]:
first_touch = first_touch.merge(journey_revenue, on="journey_id", how="left")

first_touch["attribution_model"] = "First-Touch"
first_touch["attributed_conversions"] = 1
first_touch["attributed_revenue"] = first_touch["journey_revenue"]

first_touch_output = first_touch[
    [
        "attribution_model",
        "journey_id",
        "channel",
        "campaign",
        "attributed_conversions",
        "attributed_revenue"
    ]
]

first_touch_output.head()

,attribution_model,journey_id,channel,campaign,attributed_conversions,attributed_revenue
0,First-Touch,J00001,TikTok Ads,creator_spark,1,267.15
1,First-Touch,J00004,Google Ads,brand_search,1,109.37
2,First-Touch,J00006,Referral,affiliate_blog,1,352.78
3,First-Touch,J00008,Google Ads,high_intent_keywords,1,127.44
4,First-Touch,J00011,Organic Search,comparison_page,1,89.24


In [12]:
last_touch = converted_df.sort_values("event_timestamp_utc").groupby("journey_id").last().reset_index()

last_touch[["journey_id", "channel", "campaign", "event_timestamp_utc"]].head()

,journey_id,channel,campaign,event_timestamp_utc
0,J00001,Meta Ads,retargeting,2026-02-08 18:05:00
1,J00004,Email,abandoned_cart,2026-03-21 05:31:00
2,J00006,Email,newsletter,2026-04-07 14:20:00
3,J00008,Organic Search,seo_blog,2026-01-28 08:23:00
4,J00011,Google Ads,competitor_search,2026-02-10 19:58:00


In [13]:
last_touch = last_touch.merge(journey_revenue, on="journey_id", how="left")

last_touch["attribution_model"] = "Last-Touch"
last_touch["attributed_conversions"] = 1
last_touch["attributed_revenue"] = last_touch["journey_revenue"]

last_touch_output = last_touch[
    [
        "attribution_model",
        "journey_id",
        "channel",
        "campaign",
        "attributed_conversions",
        "attributed_revenue"
    ]
]

last_touch_output.head()

,attribution_model,journey_id,channel,campaign,attributed_conversions,attributed_revenue
0,Last-Touch,J00001,Meta Ads,retargeting,1,267.15
1,Last-Touch,J00004,Email,abandoned_cart,1,109.37
2,Last-Touch,J00006,Email,newsletter,1,352.78
3,Last-Touch,J00008,Organic Search,seo_blog,1,127.44
4,Last-Touch,J00011,Google Ads,competitor_search,1,89.24


In [14]:
linear = converted_df.copy()

touchpoint_count = linear.groupby("journey_id")["event_id"].count().reset_index()

touchpoint_count = touchpoint_count.rename(
    columns={"event_id": "actual_touchpoints"}
)

touchpoint_count.head()

,journey_id,actual_touchpoints
0,J00001,4
1,J00004,4
2,J00006,4
3,J00008,5
4,J00011,2


In [15]:
linear = linear.merge(touchpoint_count, on="journey_id", how="left")
linear = linear.merge(journey_revenue, on="journey_id", how="left")

linear["attribution_model"] = "Linear"
linear["attributed_conversions"] = 1 / linear["actual_touchpoints"]
linear["attributed_revenue"] = linear["journey_revenue"] / linear["actual_touchpoints"]

linear_output = linear[
    [
        "attribution_model",
        "journey_id",
        "channel",
        "campaign",
        "attributed_conversions",
        "attributed_revenue"
    ]
]

linear_output.head()

,attribution_model,journey_id,channel,campaign,attributed_conversions,attributed_revenue
0,Linear,J00001,TikTok Ads,creator_spark,0.25,66.7875
1,Linear,J00001,Organic Search,how_to_guide,0.25,66.7875
2,Linear,J00001,Direct,direct_visit,0.25,66.7875
3,Linear,J00001,Meta Ads,retargeting,0.25,66.7875
4,Linear,J00004,Google Ads,brand_search,0.25,27.3425


In [16]:
attribution_output = pd.concat(
    [first_touch_output, last_touch_output, linear_output],
    ignore_index=True
)

attribution_output.head()

,attribution_model,journey_id,channel,campaign,attributed_conversions,attributed_revenue
0,First-Touch,J00001,TikTok Ads,creator_spark,1.0,267.15
1,First-Touch,J00004,Google Ads,brand_search,1.0,109.37
2,First-Touch,J00006,Referral,affiliate_blog,1.0,352.78
3,First-Touch,J00008,Google Ads,high_intent_keywords,1.0,127.44
4,First-Touch,J00011,Organic Search,comparison_page,1.0,89.24


In [17]:
attribution_output["attribution_model"].value_counts()

,count
attribution_model,
Linear,838
First-Touch,209
Last-Touch,209


In [18]:
attribution_output.to_csv("attribution_model_output.csv", index=False)

print("Attribution model output saved successfully")

Attribution model output saved successfully
